# Question 1: ETL Pipeline Development

### Import required libraries

In [ ]:
import cx_Oracle
from sklearn.preprocessing import MinMaxScaler
import kagglehub
import pandas as pd

c:\Users\Girish\anaconda3\envs\tf_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Extract Data from public data source (kaggle)
dataset link: https://www.kaggle.com/datasets/avish5787/housing-dataset

In [ ]:
# Download latest version
path = kagglehub.dataset_download("avish5787/housing-dataset")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\Girish\.cache\kagglehub\datasets\avish5787\housing-dataset\versions\1


In [ ]:
! copy "C:\Users\Girish\.cache\kagglehub\datasets\avish5787\housing-dataset\versions\1\Housing.csv" .\

        1 file(s) copied.


### Preview of dataset

In [ ]:
df = pd.read_csv("Housing.csv")

In [ ]:
df

,Unnamed: 0,price,lotsize,bedrooms,bathrms,stories,driveway,recroom,fullbase,gashw,airco,garagepl,prefarea
0,1,42000.0,5850,3,1,2,yes,no,yes,no,no,1,no
1,2,38500.0,4000,2,1,1,yes,no,no,no,no,0,no
2,3,49500.0,3060,3,1,1,yes,no,no,no,no,0,no
3,4,60500.0,6650,3,1,2,yes,yes,no,no,no,0,no
4,5,61000.0,6360,2,1,1,yes,no,no,no,no,0,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...
541,542,91500.0,4800,3,2,4,yes,yes,no,no,yes,0,no
542,543,94000.0,6000,3,2,4,yes,no,no,no,yes,0,no
543,544,103000.0,6000,3,2,4,yes,yes,no,no,yes,1,no
544,545,105000.0,6000,3,2,2,yes,yes,no,no,yes,1,no


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 546 entries, 0 to 545
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  546 non-null    int64  
 1   price       546 non-null    float64
 2   lotsize     546 non-null    int64  
 3   bedrooms    546 non-null    int64  
 4   bathrms     546 non-null    int64  
 5   stories     546 non-null    int64  
 6   driveway    546 non-null    object 
 7   recroom     546 non-null    object 
 8   fullbase    546 non-null    object 
 9   gashw       546 non-null    object 
 10  airco       546 non-null    object 
 11  garagepl    546 non-null    int64  
 12  prefarea    546 non-null    object 
dtypes: float64(1), int64(6), object(6)
memory usage: 55.6+ KB


### Extract data

In [ ]:
def extract_data(file_path):
    try:
        # Load data from a CSV file
        data = pd.read_csv(file_path)
        print("Data extraction successful.")
        return data
    except Exception as e:
        print(f"Error during data extraction: {e}")
        return None


[link text](https://)## Data transformation

In [ ]:
def transform_data(data):
    try:
        # dropping the first index column
        data = data.drop(data.columns[0], axis = 1)

        # Drop rows with missing values
        data = data.dropna()

        # Remove duplicates
        data = data.drop_duplicates()

        # Normalize numerical columns
        numeric_columns = data.select_dtypes(include=['float64', 'int64']).columns
        scaler = MinMaxScaler()
        data[numeric_columns] = scaler.fit_transform(data[numeric_columns])

        # Aggregation example: Group by a column and calculate mean
        # average_price_bedrooms = df.groupby('bedrooms')['price']
        # But aggregation not done here to maintain consistency

        print("Data transformation successful.")
        return data
    except Exception as e:
        print(f"Error during data transformation: {e}")
        return None

## Load the processed data into database (Oracle SQL)

In [ ]:
import cx_Oracle

def load_data_to_oracle(data, user, password, dsn, table_name):
    """
    Loads transformed data into an Oracle SQL Plus table.

    Parameters:
    - data (DataFrame): Transformed data.
    - user (str): Oracle database username.
    - password (str): Oracle database password.
    - dsn (str): Oracle Data Source Name (e.g., "//host:port/service_name").
    - table_name (str): Target table name.
    """
    try:
        # Establish connection to Oracle
        conn = cx_Oracle.connect(user, password, dsn)
        cursor = conn.cursor()

        # Dynamically create the table if it does not already exist
        column_definitions = ", ".join([
            f"{col} VARCHAR2(10)" if data[col].dtype == 'object' else f"{col} NUMBER"
            for col in data.columns
        ])
        create_table_query = f"CREATE TABLE {table_name} ({column_definitions})"
        try:
            cursor.execute(create_table_query)
        except cx_Oracle.DatabaseError as e:
            # If the table exists, an exception will occur; log the error and continue
            print(f"Table already exists or error creating table: {e}")

        # Convert DataFrame rows to tuples for insertion
        # This prepares the data for efficient batch insertion
        rows = [tuple(row) for row in data.to_numpy()]

        # Construct the insert query with placeholders
        # Using bind variables (:1, :2, ...) improves performance and prevents SQL injection
        columns = ", ".join(data.columns)
        placeholders = ", ".join([":" + str(i + 1) for i in range(len(data.columns))])
        insert_query = f"INSERT INTO {table_name} ({columns}) VALUES ({placeholders})"

        # Perform batch insert using executemany
        # Batch processing improves performance by reducing the number of database calls
        cursor.executemany(insert_query, rows)
        conn.commit()  # Commit the transaction after all rows are inserted
        print("Data loaded successfully into Oracle SQL Plus.")

    except cx_Oracle.DatabaseError as e:
        # Log any database-related errors
        print(f"Error during data loading: {e}")
    finally:
        # Ensure the cursor and connection are closed to release resources
        cursor.close()
        conn.close()


## Defining the ETL pipeline

> Add blockquote



In [ ]:
def etl_pipeline_oracle(file_path, user, password, dsn, table_name):
    # Extract
    data = extract_data(file_path)
    if data is None:
        return

    # Transform
    transformed_data = transform_data(data)
    if transformed_data is None:
        return

    # Load
    load_data_to_oracle(transformed_data, user, password, dsn, table_name)

## Pipeline Execution

### Defining inputs

In [ ]:
# Define inputs
file_path = "Housing.csv"
user = "system"
password = "xxxx"
dsn = "//localhost:1521/orcl"  # Replace with your Oracle service name
table_name = "TRANSFORMED_DATA_HOUSEPRICES"

### Run Pipeline

In [ ]:
# Execute the ETL pipeline
etl_pipeline_oracle(file_path, user, password, dsn, table_name)

Data extraction successful.
Data transformation successful.
Data loaded successfully into Oracle SQL Plus.
